In [1]:
# Core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# The star of the show
from google_play_scraper import app, reviews, Sort

In [2]:
# The unique identifier for Dashen Bank's app on the Google Play Store
DASHEN_APP_ID = 'com.dashen.dashensuperapp'

# Step 1: Get app metadata (rating, installs, description...)
app_info = app(
    DASHEN_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("DASHEN Bank App Info")
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

DASHEN Bank App Info
App Title   : Dashen Bank
Current Score: 4.2295375
Total Ratings: 5,613
Total Reviews: 1,025
Installs     : 1,000,000+


In [3]:
# Step 2: Scrape reviews
print(f"Scraping reviews for Awash Bank...")

result, continuation_token = reviews(
    DASHEN_APP_ID,
    lang='en',
    country='et',
    sort=Sort.MOST_RELEVANT,       # Most recent first
    count=500,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for Awash Bank...
Collected 500 raw reviews


In [4]:
# Let's inspect what a single raw review looks like
print("Keys in a single review:")
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 97536f33-ec3c-4169-8131-2c4fad78fb23
  userName: work done
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocJPe-mxkSg6Tx1Ep36xVjZffx-wNrD5bfDCYvwNOwBisN7bmg=mo
  content: I am experiencing a serious issue with the Dashen Super App. I visited a branch and my account was successfully activated. However, shortly after (within about 7 hours), the app started asking me again to visit the branch. In addition, my account balance is not showing even after updating the app, and I can no longer see my account number. This situation is very frustrating, especially after completing the required verification at the branch. The system does not seem stable and it is affecting
  score: 1
  thumbsUpCount: 11
  reviewCreatedVersion: 1.9.04
  at: 2026-04-16 13:45:12
  replyContent: Non

In [5]:
# Step 3: Extract only the columns we need
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'DASHEN Bank',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (500, 6)


,review_id,review,rating,date,bank,source
0,97536f33-ec3c-4169-8131-2c4fad78fb23,I am experiencing a serious issue with the Das...,1,2026-04-16 13:45:12,DASHEN Bank,Google Play
1,f04071db-8efb-4ab7-a630-23afd311e97f,I can't even get into the app just because my ...,1,2026-04-26 06:37:04,DASHEN Bank,Google Play
2,a61e3c0b-0d81-4a52-b029-8cc5e254845d,The worst mobile banking app ever... I don't t...,1,2026-04-18 01:50:55,DASHEN Bank,Google Play
3,634e5adb-d780-4107-a375-cda6beb32e8b,The app is grand. but it's missing two things....,5,2026-03-31 13:41:30,DASHEN Bank,Google Play
4,41170efe-a4b7-482c-83d1-fea6fa51993d,Very bad customer service line. they won't pic...,1,2026-05-12 07:33:07,DASHEN Bank,Google Play


In [6]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 500

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [7]:
# Rating distribution — what do users think?
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    print(f"  {int(rating)} stars: {count:>4}")

Rating distribution:
  5 stars:  315
  4 stars:   25
  3 stars:   27
  2 stars:   29
  1 stars:  104


In [8]:
# What does the date column look like right now?
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-04-16 13:45:12
1   2026-04-26 06:37:04
2   2026-04-18 01:50:55
3   2026-03-31 13:41:30
4   2026-05-12 07:33:07
5   2025-12-27 20:32:39
6   2026-02-23 09:04:51
7   2026-04-06 08:38:04
8   2026-01-30 05:09:00
9   2026-02-01 07:38:08

Date dtype: datetime64[us]


In [9]:
print("DATA QUALITY AUDIT")
print("=" * 50)

# --- Problem 1: Missing Values ---
print("\nProblem 1: Missing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Problem 1: Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [10]:
# --- Problem 2: Duplicate Reviews ---
print("Problem 2: Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Problem 2: Duplicates
------------------------------
  Exact duplicate reviews : 0
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [11]:
# --- Problem 3: Date Format ---
print("Problem 3: Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Problem 3: Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-04-16 13:45:12
  Target format: YYYY-MM-DD (string or date object)


In [12]:
# Work on a copy so raw data stays untouched
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")

Starting with: 500 reviews


In [13]:
before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

Removed 0 rows with missing critical data
Remaining: 500 reviews


In [14]:
before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

Removed 0 duplicate reviews
Remaining: 500 reviews


In [15]:
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-04-16 13:45:12
1   2026-04-26 06:37:04
2   2026-04-18 01:50:55
dtype: datetime64[us]

After normalization:
0    2026-04-16
1    2026-04-26
2    2026-04-18
dtype: str

Date range: 2025-01-13 to 2026-05-13


In [16]:
def clean_text(text):
    """Standardize review text: collapse whitespace, strip edges."""
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text

# Show before/after on a sample review
sample_raw = "  Great   app!\n\nVery useful.  "
print(f"Before: {repr(sample_raw)}")
print(f"After : {repr(clean_text(sample_raw))}")

# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")

Before: '  Great   app!\n\nVery useful.  '
After : 'Great app! Very useful.'

Removed 0 reviews that were empty after cleaning


In [17]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 500 reviews
Rating dtype: int64


In [18]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print(f"Final dataset shape: {df_clean.shape}")
df_clean.head(10)

Final dataset shape: (500, 5)


,review,rating,date,bank,source
0,very difficult app,1,2026-05-13,DASHEN Bank,Google Play
1,The application booting time is so bad..,3,2026-05-12,DASHEN Bank,Google Play
2,Very bad customer service line. they won't pic...,1,2026-05-12,DASHEN Bank,Google Play
3,good but it takes forever to load,4,2026-05-11,DASHEN Bank,Google Play
4,this app is amaizing,5,2026-05-11,DASHEN Bank,Google Play
5,best app i ever seen,5,2026-05-11,DASHEN Bank,Google Play
6,easy tp use.,5,2026-05-11,DASHEN Bank,Google Play
7,fast and very easy to use,5,2026-05-11,DASHEN Bank,Google Play
8,it is best but sometime when I need to sign in...,3,2026-05-10,DASHEN Bank,Google Play
9,nice financial app 👌,5,2026-05-10,DASHEN Bank,Google Play


In [19]:
# Save to CSV
import os
os.makedirs('data/processed', exist_ok=True)

output_path = 'data/processed/dashen_bank_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: data/processed/dashen_bank_reviews_clean.csv


In [20]:
print("  PREPROCESSING REPORT — Awash Bank Reviews")

original_count = len(df_raw)
final_count    = len(df_clean)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean['rating'].unique(), reverse=True):
    count = (df_clean['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean.columns:
    print(f"    - {col}")


  PREPROCESSING REPORT — Awash Bank Reviews

  Raw reviews collected  :    500
  Reviews after cleaning :    500
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-01-13  to  2026-05-13

  Rating distribution:
    5 stars :  315 (63.0%)  ███████████████████████████████████████████████████████████████
    4 stars :   25 ( 5.0%)  █████
    3 stars :   27 ( 5.4%)  █████
    2 stars :   29 ( 5.8%)  █████
    1 stars :  104 (20.8%)  ████████████████████

  Text length stats:
    Min    : 11 characters
    Median : 75 characters
    Max    : 500 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source
